# AIC2026 — extract keyframes + CLIP gallery

Uses `python -m tools.extract_features` from the PACs repo:

1. **extract** — videos → `VIDEO_ID/{map.csv, *.webp, embeddings.npy}` (stride + CLIP keep/drop)
2. **embed --copy-embeddings** — copy those vectors to flat `VIDEO_ID.npy` for `kis_search.py --clip-dir` (no second CLIP pass)

Edit the **paths cell** below. Example paths are Kaggle-style; change them to your dataset.

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
# Add-ons → Secrets: label must match (GitHub PAT with repo read access)
secret_value_0 = user_secrets.get_secret("AIC2026-PACs-token")

In [ ]:
!rm -rf /kaggle/working/AIC2026-PACs
!git clone https://{secret_value_0}@github.com/AkiyaNguyen/AIC2026-PACs.git

In [ ]:
!pip install -q -r /kaggle/working/AIC2026-PACs/requirements.txt

## Paths (edit these)

| Variable | Meaning | Example |
|----------|---------|---------|
| `VIDEO_DIR` | Dataset root **or** a single batch folder | `/kaggle/input/.../dataset-aic` |
| `KEYFRAMES_OUT` | Extract output root (flat `VIDEO_ID/` folders) | `/kaggle/working/keyframes-out` |
| `EMBED_OUT` | Flat CLIP gallery for search | `/kaggle/working/clip-gallery` |

You do **not** need to run once per `Videos_L21_a`, `Videos_L22`, …  
Point `VIDEO_DIR` at the **parent** dataset folder; extract recursively finds all videos under it.

Output is **flat** (not a mirror of input folders):

```text
# input (example)
/kaggle/input/.../dataset-aic/
  Videos_L21_a/
    L21_V001.mp4
    L21_V002.mp4
  Videos_L22/
    L22_V001.mp4

# after extract
/kaggle/working/keyframes-out/
  L21_V001/
    map.csv
    000000.webp
    ...
    embeddings.npy
  L21_V002/
    ...
  L22_V001/
    ...

# after embed --copy-embeddings
/kaggle/working/clip-gallery/
  L21_V001.npy
  L21_V002.npy
  L22_V001.npy
  run_meta.json
```

Turn on a **GPU** accelerator in the notebook settings before running extract.


In [ ]:
import os

# --- edit me ---
REPO = "/kaggle/working/AIC2026-PACs"

# Parent dataset root: extract finds videos in ALL child folders (Videos_L21_a, …)
VIDEO_DIR = "/kaggle/input/datasets/huhongnguyn111/dataset-aic"
# Or one batch only, e.g. VIDEO_DIR = ".../dataset-aic/Videos_L21_a"

# Extract output: KEYFRAMES_OUT/VIDEO_ID/{map.csv,*.webp,embeddings.npy}  (flat)
KEYFRAMES_OUT = "/kaggle/working/keyframes-out"

# Embed (copy) output: EMBED_OUT/VIDEO_ID.npy  → kis_search --clip-dir
EMBED_OUT = "/kaggle/working/clip-gallery"

# Extract knobs (NII-UIT defaults)
STRIDE = 10
MIN_COSINE_DISTANCE = 0.15
BATCH_SIZE = 32
DEVICE = "gpu"  # cpu | gpu | cuda

# Export so %%bash cells can read $VIDEO_DIR etc.
for k, v in {
    "REPO": REPO,
    "VIDEO_DIR": VIDEO_DIR,
    "KEYFRAMES_OUT": KEYFRAMES_OUT,
    "EMBED_OUT": EMBED_OUT,
    "STRIDE": str(STRIDE),
    "MIN_COSINE_DISTANCE": str(MIN_COSINE_DISTANCE),
    "BATCH_SIZE": str(BATCH_SIZE),
    "DEVICE": DEVICE,
}.items():
    os.environ[k] = v

print("VIDEO_DIR     =", VIDEO_DIR)
print("KEYFRAMES_OUT=", KEYFRAMES_OUT)
print("EMBED_OUT    =", EMBED_OUT)
print("DEVICE       =", DEVICE)


## 1) Extract keyframes (CLIP keep/drop)

Samples every `STRIDE`-th frame, embeds with CLIP, keeps a frame only if it is far enough from the last kept frame.

`VIDEO_DIR` may be the dataset **parent**; all nested `.mp4` (etc.) are processed in one run.  
Writes flat `KEYFRAMES_OUT/VIDEO_ID/{map.csv, *.webp, embeddings.npy}` (parent batch folder names are not kept).


In [ ]:
%%bash
set -e
cd "$REPO"
python -m tools.extract_features extract "$VIDEO_DIR" \
  --out-dir "$KEYFRAMES_OUT" \
  --stride "$STRIDE" \
  --min-cosine-distance "$MIN_COSINE_DISTANCE" \
  --batch-size "$BATCH_SIZE" \
  --device "$DEVICE"

echo "--- sample tree ---"
ls -la "$KEYFRAMES_OUT" | head
ls -la "$KEYFRAMES_OUT"/*/ 2>/dev/null | head -n 40 || true

## 2) Embed gallery via copy (recommended after extract)

`--copy-embeddings` copies each `VIDEO_ID/embeddings.npy` → `EMBED_OUT/VIDEO_ID.npy`.
No second CLIP pass. Use this for our extract tree.

Only drop `--copy-embeddings` and pass `--device gpu` if the stills have **no** `embeddings.npy` (e.g. organizer keyframes only).

In [ ]:
%%bash
set -e
cd "$REPO"
python -m tools.extract_features embed "$KEYFRAMES_OUT" \
  --out-dir "$EMBED_OUT" \
  --copy-embeddings

echo "--- gallery ---"
ls -la "$EMBED_OUT" | head

## Optional: re-encode stills with CLIP

Skip this after our extract. Uncomment only for a still tree without `embeddings.npy`.

In [ ]:
# %%bash
# set -e
# cd "$REPO"
# python -m tools.extract_features embed "$KEYFRAMES_OUT" \
#   --out-dir "$EMBED_OUT" \
#   --batch-size "$BATCH_SIZE" \
#   --device "$DEVICE"

## Sanity check one video

Confirms `map.csv` row count matches `embeddings.npy` and flat `VIDEO_ID.npy`.

In [ ]:
from pathlib import Path
import csv
import numpy as np

kf = Path(KEYFRAMES_OUT)
vid_dirs = sorted(p for p in kf.iterdir() if p.is_dir() and (p / "map.csv").is_file())
assert vid_dirs, f"No VIDEO_ID/map.csv under {kf}"
vid = vid_dirs[0]
rows = list(csv.DictReader((vid / "map.csv").open(encoding="utf-8")))
emb = np.load(vid / "embeddings.npy")
flat = Path(EMBED_OUT) / f"{vid.name}.npy"
flat_arr = np.load(flat)

print("video_id      =", vid.name)
print("map rows      =", len(rows))
print("embeddings.npy=", emb.shape)
print("flat npy      =", flat_arr.shape, flat)
assert emb.shape[0] == len(rows) == flat_arr.shape[0]
print("OK: row counts match")